In [1]:
# Block 1 — setup + read IPUMS XML schema + read raw .dat
import xml.etree.ElementTree as ET
from pyspark.sql import SparkSession, functions as F

#spark = SparkSession.builder.getOrCreate()

spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "4g") \
    .config('spark.executor.instances', 63) \
    .getOrCreate()

xml_path = "../../evargasnavarro/shared/processed/usa_00001.xml"
dat_path = "../../evargasnavarro/shared/processed/usa_00001.dat"

# Parse IPUMS DDI XML for fixed-width specs
ns = {"ddi": "ddi:codebook:2_5"}
root = ET.parse(xml_path).getroot()

specs = []
for var in root.findall(".//ddi:dataDscr/ddi:var", ns):
    loc = var.find("ddi:location", ns)
    fmt = var.find("ddi:varFormat", ns)
    specs.append({
        "name": var.attrib["name"],
        "start": int(loc.attrib["StartPos"]),
        "width": int(loc.attrib["width"]),
        "dcml": int(var.attrib.get("dcml", "0")),
        "type": fmt.attrib.get("type", "character") if fmt is not None else "character"
    })

raw_df = spark.read.text(dat_path)
raw_df.show(5, truncate=False)  # raw fixed-width lines
print(f"Columns in XML spec: {len(specs)}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [2]:
# Block 2 — parse to Spark DataFrame + cast numerics + inspect
df = raw_df.select(*[
    F.substring("value", s["start"], s["width"]).alias(s["name"])
    for s in specs
])

for s in specs:
    if s["type"] == "numeric":
        df = df.withColumn(s["name"], F.col(s["name"]).cast("double"))
        if s["dcml"] > 0:
            df = df.withColumn(s["name"], F.col(s["name"]) / (10 ** s["dcml"]))

# readable checks
print("rows:", df.count())
print("cols:", len(df.columns))
df.select("YEAR", "STATEFIP", "SEX", "AGE", "RACE", "EDUC", "INCTOT").show(10, truncate=False)
df.printSchema()

rows: 67125780
cols: 238
+------+--------+---+----+----+----+---------+
|YEAR  |STATEFIP|SEX|AGE |RACE|EDUC|INCTOT   |
+------+--------+---+----+----+----+---------+
|2001.0|1.0     |1.0|39.0|1.0 |3.0 |6400.0   |
|2001.0|1.0     |1.0|33.0|2.0 |10.0|30000.0  |
|2001.0|1.0     |2.0|40.0|1.0 |6.0 |16500.0  |
|2001.0|1.0     |1.0|10.0|1.0 |1.0 |9999999.0|
|2001.0|1.0     |2.0|21.0|1.0 |3.0 |0.0      |
|2001.0|1.0     |1.0|39.0|1.0 |6.0 |21000.0  |
|2001.0|1.0     |2.0|55.0|1.0 |10.0|12000.0  |
|2001.0|1.0     |1.0|38.0|1.0 |3.0 |20000.0  |
|2001.0|1.0     |2.0|32.0|1.0 |2.0 |0.0      |
|2001.0|1.0     |1.0|10.0|1.0 |1.0 |9999999.0|
+------+--------+---+----+----+----+---------+
only showing top 10 rows

root
 |-- YEAR: double (nullable = true)
 |-- SAMPLE: double (nullable = true)
 |-- SERIAL: double (nullable = true)
 |-- CBSERIAL: double (nullable = true)
 |-- NUMPREC: double (nullable = true)
 |-- SUBSAMP: double (nullable = true)
 |-- HHWT: double (nullable = true)
 |-- EXPWTH: double 

# Data Exploration using Spark

In [3]:
print("Number of Rows:", df.count())
print("Number of Columns:", len(df.columns))

Number of Rows: 67125780
Number of Columns: 238


From the schema printed above, REPWT1-80 are known as replicated weights which are only necessary for generating empirically derived standard errors. There is no reason to include them in data exploration.

In [4]:
target_columns = [c for c in df.columns if "REPWT" not in c]
df_target = df.select(target_columns)

### Columns of Interest

In [5]:
df_target.describe().show(vertical=True, truncate=False)

-RECORD 0----------------------------
 summary     | count                 
 YEAR        | 67125780              
 SAMPLE      | 67125780              
 SERIAL      | 67125780              
 CBSERIAL    | 62469664              
 NUMPREC     | 67125780              
 SUBSAMP     | 67125780              
 HHWT        | 67125780              
 EXPWTH      | 5880607               
 HHTYPE      | 67125780              
 CBHHTYPE    | 19335281              
 CLUSTER     | 67125780              
 ADJUST      | 67125780              
 CPI99       | 67125780              
 REGION      | 67125780              
 STATEICP    | 67125780              
 STATEFIP    | 67125780              
 COUNTYICP   | 62469664              
 COUNTYFIP   | 62469664              
 PUMA        | 62469664              
 PUMASUPR    | 21047877              
 CONSPUMA    | 21047877              
 CPUMA0010   | 52267589              
 CPUMA1020   | 41421787              
 DENSITY     | 62469664              
 METRO      

### Counting Nulls

In [8]:
all_null_counts = df_target.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in df_target.columns])
all_null_counts_dict = all_null_counts.first().asDict()

In [9]:
for k, v in all_null_counts_dict.items():
    if v != 0:
        print(f"{k}: {v}")

CBSERIAL: 4656116
EXPWTH: 61245173
CBHHTYPE: 47790499
COUNTYICP: 4656116
COUNTYFIP: 4656116
PUMA: 4656116
PUMASUPR: 46077903
CONSPUMA: 46077903
CPUMA0010: 14858191
CPUMA1020: 25703993
DENSITY: 4656116
METRO: 4656116
PCTMETRO: 25703993
METAREA: 44882975
METAREAD: 44882975
MET2013: 4656116
MET2013ERR: 4656116
MET2023: 25703993
MET2023ERR: 25703993
METPOP00: 46077903
METPOP10: 4656116
METPOP20: 25703993
CITY: 4656116
CITYERR: 4656116
CITYPOP: 4656116
HOMELAND: 4656116
APPAL: 46077903
APPALD: 46077903
MET2003: 65930852
RESPMODE: 4656116
AGEORIG: 58888377
BIRTHQTR: 4656116
MARRNO: 13498899
MARRINYR: 13498899
YRMARR: 13498899
DIVINYR: 13498899
WIDINYR: 13498899
YRNATUR: 13498899
RACHSING: 10202075
PREDAI: 10202075
PREDAPI: 10202075
PREDBLK: 10202075
PREDWHT: 10202075
PREDHISP: 10202075
DEGFIELD: 16499556
DEGFIELDD: 16499556
DEGFIELD2: 16499556
DEGFIELD2D: 16499556
WKSWORK1: 34291600
MIGCOUNTY1: 4656116
MIGPUMA1: 4656116
MIGPUMANOW: 4656116
MIGSPUMA1: 46077903
MIGMETAREA1: 46077903
MIGMET131:

### Unique Values

In [11]:
distinct_counts = df_target.select().distinct().count()
distinct_counts_dict = distinct_counts.first().asDict()

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
for k, v in distinct_counts_dict.items():
    if v != 0:
        print(f"{k}: {v}")

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
